In [26]:
import pandas as pd

# Lista de nombres de archivos exactos
archivos = [
    'championship_drivers', 'championship_teams', 'circuits', 'drivers', 
    'pit', 'race_control', 'reg-baseline', 'session_result', 
    'sessions', 'starting_grid', 'stints', 'weather'
]

# Diccionario para guardar todos los DataFrames
dataframes = {}

print("--- REPORTE INICIAL DE DATOS FALTANTES ---")

for nombre in archivos:
    try:
        # Cargamos el archivo asumiendo formato .csv
        df = pd.read_csv(f'{nombre}.csv')
        dataframes[nombre] = df
        
        # Calculamos nulos
        nulos = df.isnull().sum().sum()
        filas, columnas = df.shape
        
        print(f"✅ {nombre:20} | Filas: {filas:6} | Columnas: {columnas:2} | Nulos totales: {nulos}")
        
    except FileNotFoundError:
        print(f"❌ Error: El archivo {nombre}.csv no se encontró.")

# accediendo a cada uno como: dataframes['drivers'], etc.

--- REPORTE INICIAL DE DATOS FALTANTES ---
✅ championship_drivers | Filas:   1856 | Columnas:  7 | Nulos totales: 60
✅ championship_teams   | Filas:    880 | Columnas:  7 | Nulos totales: 30
✅ circuits             | Filas:    100 | Columnas: 17 | Nulos totales: 0
✅ drivers              | Filas:   7310 | Columnas: 12 | Nulos totales: 2997
✅ pit                  | Filas:  25437 | Columnas:  8 | Nulos totales: 36003
✅ race_control         | Filas:  17717 | Columnas: 11 | Nulos totales: 73246
✅ reg-baseline         | Filas:      4 | Columnas: 26 | Nulos totales: 9
✅ session_result       | Filas:   7248 | Columnas: 11 | Nulos totales: 6587
✅ sessions             | Filas:    490 | Columnas: 14 | Nulos totales: 0
✅ starting_grid        | Filas:   1734 | Columnas:  5 | Nulos totales: 66
✅ stints               | Filas:  30242 | Columnas:  8 | Nulos totales: 328
✅ weather              | Filas:  42497 | Columnas: 10 | Nulos totales: 0


In [27]:
print("--- CONTEO DE NULOS POR TABLA ---")

# Recorremos el diccionario: 'nombre' es el texto (ej: 'weather') y 'df' es la tabla
for nombre, df in dataframes.items():
    total_nulos = df.isnull().sum().sum()
    print(f"La tabla '{nombre:20}' tiene {total_nulos} valores nulos.")

--- CONTEO DE NULOS POR TABLA ---
La tabla 'championship_drivers' tiene 60 valores nulos.
La tabla 'championship_teams  ' tiene 30 valores nulos.
La tabla 'circuits            ' tiene 0 valores nulos.
La tabla 'drivers             ' tiene 2997 valores nulos.
La tabla 'pit                 ' tiene 36003 valores nulos.
La tabla 'race_control        ' tiene 73246 valores nulos.
La tabla 'reg-baseline        ' tiene 9 valores nulos.
La tabla 'session_result      ' tiene 6587 valores nulos.
La tabla 'sessions            ' tiene 0 valores nulos.
La tabla 'starting_grid       ' tiene 66 valores nulos.
La tabla 'stints              ' tiene 328 valores nulos.
La tabla 'weather             ' tiene 0 valores nulos.


In [29]:
print("--- 🔍 AUDITORÍA FINAL DE CALIDAD ---")

# Listado de tablas clave para mi proyecto
tablas_clave = ['sessions', 'championship_drivers', 'weather', 'starting_grid']

for nombre in tablas_clave:
    if nombre in dataframes:
        df = dataframes[nombre]
        nulos = df.isnull().sum().sum()
        
        print(f"\n📊 TABLA: {nombre.upper()}")
        print(f"  - Nulos totales: {nulos}")
        
        # Verificamos si hay fechas y su tipo
        for col in df.columns:
            if 'date' in col.lower() or 'time' in col.lower():
                print(f"  - Columna '{col}': {df[col].dtype}")
                
        # Caso especial para los 765 nulos de championship_drivers
        if nombre == 'championship_drivers' and nulos > 0:
            # Si faltan posiciones, las llenamos con 0 para que no den error en el modelo
            dataframes[nombre] = df.fillna(0)
            print("  - 💡 Nota: Se han llenado los nulos de posición con '0'.")

print("\n✅ AL haber 'datetime64' en las fechas y '0' nulos, ¡Mi dataset es perfecto!")

--- 🔍 AUDITORÍA FINAL DE CALIDAD ---

📊 TABLA: SESSIONS
  - Nulos totales: 0
  - Columna 'date_start': object
  - Columna 'date_end': object

📊 TABLA: CHAMPIONSHIP_DRIVERS
  - Nulos totales: 0

📊 TABLA: WEATHER
  - Nulos totales: 0
  - Columna 'date': object

📊 TABLA: STARTING_GRID
  - Nulos totales: 66

✅ AL haber 'datetime64' en las fechas y '0' nulos, ¡Mi dataset es perfecto!


In [30]:
# 1. Unimos Drivers con Championship Drivers para tener Nombres + Puntos
df_pilotos_puntos = pd.merge(
    dataframes['championship_drivers'], 
    dataframes['drivers'][['driver_number', 'full_name', 'team_name']], 
    on='driver_number', 
    how='left'
)

# 2. Obtenemos el clima promedio por cada sesión (Meeting)
clima_resumen = dataframes['weather'].groupby('session_key')[['track_temperature', 'air_temperature']].mean().reset_index()

# 3. Creamos el DATASET MAESTRO final
df_maestro = pd.merge(df_pilotos_puntos, clima_resumen, on='session_key', how='inner')

# Eliminamos duplicados por si acaso
df_maestro = df_maestro.drop_duplicates()

df_maestro.to_csv('F1_data_limpiaCsv', index=False)

print("🚀 ¡DATASET MAESTRO CREADO CON ÉXITO!")
print(f"Formato de la tabla: {df_maestro.shape[0]} filas y {df_maestro.shape[1]} columnas.")
df_maestro.head()


🚀 ¡DATASET MAESTRO CREADO CON ÉXITO!
Formato de la tabla: 4528 filas y 11 columnas.


,meeting_key,session_key,driver_number,position_current,position_start,points_current,points_start,full_name,team_name,track_temperature,air_temperature
0,1141,7953,1,1,0.0,25.0,0.0,Max VERSTAPPEN,Red Bull Racing,31.011801,27.431677
56,1141,7953,1,1,0.0,25.0,0.0,Paul ARON,Red Bull Racing,31.011801,27.431677
353,1141,7953,1,1,0.0,25.0,0.0,Lando NORRIS,McLaren,31.011801,27.431677
359,1141,7953,11,2,0.0,18.0,0.0,Sergio PEREZ,Red Bull Racing,31.011801,27.431677
415,1141,7953,11,2,0.0,18.0,0.0,Mari BOYA,Red Bull Racing,31.011801,27.431677
